### Implement a policy gradient algorithms like REINFORCE or Proximal Policy Optimization (PPO) to solve a continuous control task.

In [32]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [33]:
# Continuous control environment
env = gym.make("MountainCarContinuous-v0")

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]  # continuous

print("State:", state_dim, "Action:", action_dim)

State: 2 Action: 1


## Policy Network (Gaussian Policy)

In [34]:
# Policy outputs mean and std for continuous action

class Policy(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        self.mean = nn.Linear(64, action_dim)   # mean of action
        self.log_std = nn.Parameter(torch.zeros(action_dim))  # std

    def forward(self, state):
        x = self.net(state)

        mean = self.mean(x)
        std = torch.exp(self.log_std)  # ensure positive

        return mean, std

In [35]:
policy = Policy()
optimizer = optim.Adam(policy.parameters(), lr=1e-3)

gamma = 0.99  # discount factor

## Action Sampling

In [36]:
def get_action(state):
    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)

    mean, std = policy(state)

    dist = torch.distributions.Normal(mean, std)

    action = dist.sample()
    log_prob = dist.log_prob(action).sum()

    action = action.squeeze(0).detach()

    return action, log_prob

## Compute Returns

In [37]:
# Compute discounted rewards
def step_env(action):
    # Convert tensor → python float safely (NO numpy)
    action = action.item()

    # Gym expects array-like
    next_state, reward, done, _, _ = env.step([action])

    return next_state, reward, done

## Training Loop (REINFORCE)

In [38]:
episodes = 50

for ep in range(episodes):
    state, _ = env.reset()

    log_probs = []
    rewards = []
    total_reward = 0
    done = False

    while not done:
        action, log_prob = get_action(state)

        next_state, reward, done = step_env(action)

        log_probs.append(log_prob)
        rewards.append(reward)

        state = next_state
        total_reward += reward

    # returns
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns)

    loss = 0
    for log_prob, G in zip(log_probs, returns):
        loss += -log_prob * G

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Episode {ep+1} | Reward: {total_reward:.2f}")

Episode 1 | Reward: -2998.04
Episode 2 | Reward: -3869.42
Episode 3 | Reward: -5813.43
Episode 4 | Reward: -1015.07
Episode 5 | Reward: -270.10
Episode 6 | Reward: -3719.47
Episode 7 | Reward: -211.00
Episode 8 | Reward: -271.04
Episode 9 | Reward: -399.21
Episode 10 | Reward: -462.45
Episode 11 | Reward: -632.55
Episode 12 | Reward: -17.70
Episode 13 | Reward: -318.90
Episode 14 | Reward: -315.86
Episode 15 | Reward: -285.19
Episode 16 | Reward: -223.24
Episode 17 | Reward: -162.57
Episode 18 | Reward: -202.94
Episode 19 | Reward: -48.83
Episode 20 | Reward: 6.11
Episode 21 | Reward: -54.30
Episode 22 | Reward: -230.25
Episode 23 | Reward: -166.09
Episode 24 | Reward: -267.59
Episode 25 | Reward: -90.70
Episode 26 | Reward: -319.25
Episode 27 | Reward: -111.21
Episode 28 | Reward: -210.17
Episode 29 | Reward: -216.58
Episode 30 | Reward: -356.12
Episode 31 | Reward: -66.12
Episode 32 | Reward: 31.04
Episode 33 | Reward: -129.44
Episode 34 | Reward: -55.52
Episode 35 | Reward: -345.57
